Figure S14: GWAS colocalizations across tissues and traits. The total number of GWAS hits near clusters is the number of unique GWAS lead variants fine-mapped within 1MB of a gene cluster. The number of GWAS hits colocalized per GWAS trait in each tissue is given for eQTLs and for novel pcQTLs (pcQTLs not colocalized by an eQTL).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns 
import statsmodels.api as sm
from tqdm.auto import tqdm
tqdm.pandas()

# get outputs from a config file
import yaml
config_path= '/home/klawren/oak/pcqtls/config_old/main_pcqtl.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

import sys
sys.path.append(f'{config["working_dir"]}/{config["code_dir"]}')
from utils import *
from group_signals import load_gwas_coloc

# Set up plotting
plt.rcParams.update({'font.size': 10})
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42

# Load tissue data
tissue_df = load_tissue_df(config)
tissue_ids = load_tissue_ids(config)

Using Python: /home/klawren/.pixi/envs/python/bin/python


In [3]:
gwas_signal_groups = load_across_tissues(config, load_gwas_signal_groups)
gwas_signal_groups['phenotype_id'] = gwas_signal_groups['signal_id'].str.split('-')
gwas_signal_groups_explode = gwas_signal_groups.explode('phenotype_id')

gwas_hits = gwas_signal_groups_explode[gwas_signal_groups_explode['phenotype_id'].str.contains('gwas')]
gwas_hits['gwas_trait'] = gwas_hits['phenotype_id'].str.split('_cluster').str[0].str.strip('gwas_')

/local/scratch/klawren/slrmtmp.49273183/ipykernel_8704/4205081416.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gwas_hits['gwas_trait'] = gwas_hits['phenotype_id'].str.split('_cluster').str[0].str.strip('gwas_')


In [26]:
gwas_coloc = load_gwas_coloc(config)

getting files in directory: /home/klawren/oak/pcqtls/output/proteincoding_main/coloc/gwas_susie_True
found 1482 files
File is empty: /home/klawren/oak/pcqtls/output/proteincoding_main/coloc/gwas_susie_True/Skin_Not_Sun_Exposed_Suprapubic/Skin_Not_Sun_Exposed_Suprapubic.v8.GEFOS_Forearm.gwas_coloc.txt
File is empty: /home/klawren/oak/pcqtls/output/proteincoding_main/coloc/gwas_susie_True/Skin_Not_Sun_Exposed_Suprapubic/Skin_Not_Sun_Exposed_Suprapubic.v8.GPC-NEO-NEUROTICISM.gwas_coloc.txt
File is empty: /home/klawren/oak/pcqtls/output/proteincoding_main/coloc/gwas_susie_True/Skin_Not_Sun_Exposed_Suprapubic/Skin_Not_Sun_Exposed_Suprapubic.v8.EGG_Pubertal_growth_10F.gwas_coloc.txt
File is empty: /home/klawren/oak/pcqtls/output/proteincoding_main/coloc/gwas_susie_True/Skin_Not_Sun_Exposed_Suprapubic/Skin_Not_Sun_Exposed_Suprapubic.v8.PGC_ASD_2017_CEU.gwas_coloc.txt
File is empty: /home/klawren/oak/pcqtls/output/proteincoding_main/coloc/gwas_susie_True/Skin_Not_Sun_Exposed_Suprapubic/Skin_No

In [36]:
num_gwas_hits_tissue = gwas_coloc[['gwas_id', 'hit1', 'tissue_id']].drop_duplicates().groupby(['gwas_id', 'tissue_id']).size().reset_index(name='num_hits')
num_gwas_hits_total = gwas_coloc[['gwas_id', 'hit1']].drop_duplicates().groupby(['gwas_id']).size().reset_index(name='num_hits')


In [60]:
# Calculate total GWAS hits per tissue
num_gwas_hits = gwas_coloc[['gwas_id', 'hit1', 'tissue_id']].drop_duplicates().groupby(['gwas_id', 'tissue_id']).size().reset_index(name='num_hits')

# Calculate total GWAS hits across all tissues for each GWAS (for the leftmost panel)
num_gwas_hits_total = gwas_coloc[['gwas_id', 'hit1']].drop_duplicates().groupby(['gwas_id']).size().reset_index(name='num_hits')
num_gwas_hits_total = num_gwas_hits_total.set_index('gwas_id')

# Get full set of GWAS trait IDs for consistent indexing
all_gwas_traits = sorted(num_gwas_hits['gwas_id'].unique())

# Panel 1: eQTL-colocalized GWAS Hits
trait_tissue_counts_eqtl = gwas_hits[gwas_hits['num_e_coloc']>0].groupby(['gwas_trait', 'tissue_id']).size().reset_index(name='count')
trait_tissue_matrix_eqtl = trait_tissue_counts_eqtl.pivot(index='gwas_trait', columns='tissue_id', values='count').fillna(0).astype(int)

# Panel 2: Novel pcQTL-colocalized GWAS Hits
trait_tissue_counts_pcqtl = gwas_hits[gwas_hits['num_e_coloc']==0].groupby(['gwas_trait', 'tissue_id']).size().reset_index(name='count')
trait_tissue_matrix_pcqtl = trait_tissue_counts_pcqtl.pivot(index='gwas_trait', columns='tissue_id', values='count').fillna(0).astype(int)

# Union of all tissues for shared axes
all_tissues = sorted(set(trait_tissue_matrix_eqtl.columns).union(trait_tissue_matrix_pcqtl.columns))

# Reindex both matrices to have gwas_trait (rows) exactly matching gwas_id from num_gwas_hits, and all tissues as columns
trait_tissue_matrix_eqtl = trait_tissue_matrix_eqtl.reindex(index=all_gwas_traits, columns=all_tissues, fill_value=0).astype(int)
trait_tissue_matrix_pcqtl = trait_tissue_matrix_pcqtl.reindex(index=all_gwas_traits, columns=all_tissues, fill_value=0).astype(int)

# Sort rows by total GWAS hits (num_gwas_hits_total)
# (If there are ties, the order within ties will follow the default .sort_values())
gwas_order = num_gwas_hits_total['num_hits'].sort_values(ascending=False).index

trait_tissue_matrix_eqtl = trait_tissue_matrix_eqtl.loc[gwas_order]
trait_tissue_matrix_pcqtl = trait_tissue_matrix_pcqtl.loc[gwas_order]

# Create gwas_id x tissue_id heatmap matrix and reindex/align as above
gwas_tissue_matrix = num_gwas_hits.pivot(index='gwas_id', columns='tissue_id', values='num_hits').fillna(0).astype(int)
gwas_tissue_matrix = gwas_tissue_matrix.reindex(index=gwas_order, columns=all_tissues, fill_value=0).astype(int)

# Reindex the leftmost total hits panel to match the sorting of the main gwas_id x tissue_id matrix
num_gwas_hits_total_ordered = num_gwas_hits_total.reindex(gwas_order).fillna(0).astype(int)

# Set up the four-panel figure
fig, axes = plt.subplots(
    1, 4, 
    figsize=(22, max(4, 0.4 * max(len(gwas_order), len(gwas_order)))), 
    gridspec_kw={'wspace':0.05, 'width_ratios':[0.5, 2, 2, 2]}, 
    sharey=True, dpi=300
)

ax_total, ax_left, ax0, ax1 = axes

# Panel 0: Total GWAS hits across tissues (one-column heatmap)
sns.heatmap(
    num_gwas_hits_total_ordered,
    annot=True,
    fmt='d',
    cmap='Blues',
    cbar=False,
    linewidths=0.2,
    ax=ax_total,
    yticklabels=[idx.replace('_', ' ') for idx in num_gwas_hits_total_ordered.index],
    xticklabels=[" "]
)
ax_total.set_title('GWAS hits near clusters\n(all tissues)', pad=25)
ax_total.xaxis.set_ticks_position('top')
ax_total.xaxis.set_label_position('top')
ax_total.set_xlabel('')
ax_total.set_ylabel('')

# Left panel: Total GWAS hits (gwas_id x tissue_id)
sns.heatmap(
    gwas_tissue_matrix,
    annot=True,
    fmt='d',
    cmap='Blues',
    cbar=False,
    linewidths=0.2,
    ax=ax_left,
    xticklabels=[lbl.replace('_', ' ') for lbl in gwas_tissue_matrix.columns]
)
ax_left.set_title('GWAS hits near clusters\n(per tissue)', pad=30)
ax_left.xaxis.set_ticks_position('top')
ax_left.xaxis.set_label_position('top')
ax_left.set_xlabel('')
ax_left.set_ylabel('')
ax_left.set_xticklabels(ax_left.get_xticklabels(), rotation=45, ha='left')

# Middle panel: eQTL-colocalized GWAS Hits
sns.heatmap(
    trait_tissue_matrix_eqtl,
    annot=True,
    fmt='d',
    cmap='Blues',
    cbar=False,
    linewidths=0.2,
    ax=ax0,
    xticklabels=[lbl.replace('_', ' ') for lbl in trait_tissue_matrix_eqtl.columns]
)
ax0.set_title('GWAS hits colocalized with eQTLs', pad=30)
ax0.xaxis.set_ticks_position('top')
ax0.xaxis.set_label_position('top')
ax0.set_xlabel('')
ax0.set_ylabel('')
ax0.set_xticklabels(ax0.get_xticklabels(), rotation=45, ha='left')

# Right panel: Novel pcQTL-colocalized GWAS Hits
sns.heatmap(
    trait_tissue_matrix_pcqtl,
    annot=True,
    fmt='d',
    cmap='Blues',
    cbar=False,
    linewidths=0.2,
    ax=ax1,
    xticklabels=[lbl.replace('_', ' ') for lbl in trait_tissue_matrix_pcqtl.columns]
)
ax1.set_title('GWAS hits colocalized with novel pcQTLs', pad=30)
ax1.xaxis.set_ticks_position('top')
ax1.xaxis.set_label_position('top')
ax1.set_xlabel('')
ax1.set_ylabel('')
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha='left')

ax_total.set_yticklabels([str(lbl.get_text()).replace('_', ' ') for lbl in ax_total.get_yticklabels()], rotation=0)

plt.savefig(f"{config['working_dir']}/workflow/supplemental_figures/figures/figure_s14.pdf", 
            transparent=True, bbox_inches='tight', dpi=300)
plt.show()